# Sprint 11: Ground Truth Target Encoding Demonstration

This notebook demonstrates the implementation of the dense 3D heatmap target generator using ground truth coordinates.

## Key Features:
- 3D Gaussian heatmap generation centered on cell coordinates
- Configurable `sigma` and `radius` parameters (loaded from Hydra dataset configs or passed directly)
- Proper clipping at volume boundaries
- Merging of overlapping Gaussians using the maximum value operator
- Strictly normalized outputs in $[0, 1]$

In [ ]:
import torch
import matplotlib.pyplot as plt
from src.preprocessing.target_generator import TargetGenerator

# Set random seed for reproducibility
torch.manual_seed(42)

## 1. Setup Parameters and Instantiate TargetGenerator

We initialize a `TargetGenerator` with a volume shape of `(8, 64, 64)`. We will explicitly specify `sigma=2.0` and `radius=4.0` for visualization purposes.

In [ ]:
volume_shape = (8, 64, 64)
generator = TargetGenerator(volume_shape=volume_shape, sigma=2.0, radius=4.0)

print(f"Volume Shape: {generator.volume_shape}")
print(f"Sigma: {generator.sigma}")
print(f"Radius: {generator.radius}")

## 2. Generate Coordinates

We will define 5 coordinates to demonstrate various features:
- Coordinates 0 and 1 will be close to each other to demonstrate maximum value merging.
- Coordinate 2 will be close to the boundary to demonstrate clipping.
- Coordinates 3 and 4 will be random points.

In [ ]:
# Layout [z, y, x]
coords = torch.tensor([
    [4.0, 32.0, 32.0],  # Point 1 (Center)
    [4.0, 33.5, 33.5],  # Point 2 (Overlapping with Point 1)
    [0.2, 5.0, 62.0],   # Point 3 (Border case - near Z-boundary and X-boundary)
    [2.0, 15.0, 45.0],  # Point 4
    [6.0, 50.0, 10.0],  # Point 5
], dtype=torch.float32)

print("Generated Coordinates:")
print(coords)

## 3. Generate Heatmap Target

We run the generator to obtain the 3D dense heatmap target.

In [ ]:
heatmap = generator.generate(coords)

print(f"Generated heatmap shape: {heatmap.shape}")
print(f"Heatmap range: [{heatmap.min().item():.4f}, {heatmap.max().item():.4f}]")
print(f"Number of non-zero voxels: {torch.sum(heatmap > 0.0).item()}")

## 4. Visualize Slices

We plot the Z-slices of the generated 3D heatmap to observe the Gaussians, boundary clipping, and overlapping merge behavior.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Extract the heatmap tensor (Z, Y, X)
heatmap_3d = heatmap[0].cpu().numpy()

for z in range(volume_shape[0]):
    ax = axes[z]
    im = ax.imshow(heatmap_3d[z], cmap='hot', vmin=0, vmax=1)
    ax.set_title(f"Z-Slice {z}")
    ax.axis('off')
    
    # Overlay coords falling within this Z slice (rounded to nearest integer Z)
    slice_coords = coords[torch.round(coords[:, 0]) == z]
    for c in slice_coords:
        ax.plot(c[2].item(), c[1].item(), 'bx', markersize=8, label='GT')

plt.tight_layout()
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax)
plt.suptitle("Target Heatmap Z-Slices (Hot Colormap) with Ground Truth Cell Coordinates Overlay (blue 'x')", y=0.98, fontsize=16)
plt.show()